# Lesson 7 — Ship your predictive-text engine

Everything from Lessons 2–6 goes into one `NgramLM` class with the usual ML shape: `fit` on text, then
`suggest_next`, `generate` and `perplexity`. The same code also lives in `ngram_lm.py` so I can import it later.

In [1]:
import re, math, random
from collections import Counter, defaultdict

In [2]:
def to_sentences(text):
    return [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]

def to_tokens(sentence):
    return re.findall(r"[a-z']+", sentence.lower())

In [3]:
def build(sents, n):
    """context (n-1 words) -> Counter of the words that follow it."""
    foll = defaultdict(Counter)
    for s in sents:
        toks = ["<s>"] * (n - 1) + to_tokens(s) + ["</s>"]
        for i in range(len(toks) - n + 1):
            foll[tuple(toks[i:i + n - 1])][toks[i + n - 1]] += 1
    return foll

## The class

- `fit` splits the text into sentences, counts n-grams with `build`, and records the vocabulary.
- `_prob` is add-k smoothing (k = 0.5 by default, gentler than add-1).
- `suggest_next` takes a whole typed phrase, keeps its last n-1 tokens as the context, and returns the most
  frequent followers. It never suggests `</s>`, and falls back to `(no idea)`.
- `generate` is the Lesson 4 sampling loop. `perplexity` is the Lesson 6 score.

In [4]:
class NgramLM:
    "A smoothed n-gram language model: fit -> suggest_next / generate / perplexity."
    def __init__(self, n=2, k=0.5):
        self.n, self.k = n, k

    def fit(self, text):
        self.sents = to_sentences(text)
        self.foll = build(self.sents, self.n)
        self.V = set(t for s in self.sents for t in to_tokens(s)) | {"</s>"}
        return self

    def _prob(self, ctx, w):
        c = self.foll.get(ctx, Counter()); total = sum(c.values())
        return (c[w] + self.k) / (total + self.k * len(self.V))

    def suggest_next(self, text, topk=3):
        ctx = tuple((["<s>"] * (self.n - 1) + to_tokens(text))[-(self.n - 1):])
        c = self.foll.get(ctx, Counter())
        words = [w for w, _ in c.most_common() if w != "</s>"]   # never suggest end-of-sentence
        return words[:topk] or ["(no idea)"]

    def generate(self, seed=0, max_len=25):
        rng = random.Random(seed); ctx = ("<s>",) * (self.n - 1); out = []
        for _ in range(max_len):
            c = self.foll.get(ctx)
            if not c: break
            words, wts = zip(*c.items())
            nxt = rng.choices(words, weights=wts)[0]
            if nxt == "</s>": break
            out.append(nxt); ctx = (ctx + (nxt,))[-(self.n - 1):]
        return " ".join(out)

    def perplexity(self, text):
        log_sum, N = 0.0, 0
        for s in to_sentences(text):
            toks = ["<s>"] * (self.n - 1) + to_tokens(s) + ["</s>"]
            for i in range(len(toks) - self.n + 1):
                ctx, w = tuple(toks[i:i + self.n - 1]), toks[i + self.n - 1]
                log_sum += math.log(self._prob(ctx, w)); N += 1
        return math.exp(-log_sum / N)

## Using it

A bigram model fitted on the corpus, then a few phrases typed into the "keyboard", and one generated sentence.

In [5]:
lm = NgramLM(n=2, k=0.5).fit(open("corpus.txt").read())
for phrase in ["i drink", "the morning", "on the", "she reads a"]:
    print(f"{phrase!r:15s}->  {lm.suggest_next(phrase)}")
print("\ngenerated:", lm.generate(seed=3))

'i drink'      ->  ['coffee', 'tea']
'the morning'  ->  ['train', 'coffee', 'before']
'on the'       ->  ['morning', 'weekend', 'park']
'she reads a'  ->  ['good', 'book', 'film']

generated: the quiet and coffee


Because it's a bigram, only the last word counts: "the morning" is treated as "morning", which is why
`</s>` (its top follower) had to be filtered out. Seed 3 gives the same sentence as the Lesson 4 bigram did,
since it's the same counts and the same sampling loop.

## The reusable module

`ngram_lm.py` holds the same functions and class. This check imports it and confirms it behaves exactly like
the notebook version (no output means it passed).

In [6]:
import ngram_lm
lm2 = ngram_lm.NgramLM(n=2, k=0.5).fit(open("corpus.txt").read())
for phrase in ["i drink", "the morning", "on the", "she reads a"]:
    assert lm2.suggest_next(phrase) == lm.suggest_next(phrase)
assert lm2.generate(seed=3) == lm.generate(seed=3)
assert abs(lm2.perplexity(open("corpus.txt").read()) - lm.perplexity(open("corpus.txt").read())) < 1e-12

## Test yourself

**1. Why does `suggest_next` filter out `</s>`?**
`</s>` is a real prediction ("the sentence ends here"), but a keyboard should only offer words. After
"morning", `</s>` is the top follower, so without the filter the first suggestion would be useless.

**2. What context does `suggest_next("she reads a")` use with n=2?**
Only the last token, `("a",)`. The bigram ignores "she reads", which is why it suggests "good", "book" and
"film" (words that follow "a" anywhere in the corpus).

**3. Why is `_prob` smoothed but `suggest_next` and `generate` use raw counts?**
Ranking and sampling only need relative counts among words actually seen, and smoothing wouldn't change the
ranking of seen words. Perplexity has to score *any* word, including unseen ones, so it needs a non-zero
probability.

## My notes

The key idea: a full language model fits behind four methods (fit, suggest, generate, score), the same interface
as any ML model. What surprised me is that seed 3 reproduces the Lesson 4 bigram sentence word for word. The
class really is just the earlier lessons packaged up.